# Matched Gemma-2-9B-it APPS replication with Gemma Scope

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex

This notebook launches the prospectively frozen second-model replication. It keeps the APPS rows, split, prompts, controller, budgets, and prevalence sweep from the Qwen study fixed. It adds one official Gemma Scope SAE-feature observer.

The workflow permits exactly one compatibility smoke before review. The full A100 run remains disabled until `RUN_FULL` is changed explicitly after code review. The smoke reports shapes and compatibility only; it does not fit or rank observers.

## 1. Accelerator and model access

Select an A100 runtime. Before continuing, accept the Gemma license on Hugging Face for `google/gemma-2-9b-it`. The login cell uses your Hugging Face account token and does not store it in the notebook.

In [ ]:
# Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex
import subprocess, sys
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
from huggingface_hub import login
login()

## 2. Install the frozen ObserverBench source

The source manifest will reject a modified config, runner, measurement module, test, or notebook. If the repository is still private, authenticate Git in the usual Colab-safe way before running the clone.

In [ ]:
from pathlib import Path
REPO = Path("/content/observerbench")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/kwisatzh/observerbench.git", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[ai-control]"], check=True)
CONFIG = REPO / "configs/revision/ai_control/controlarena_apps_gemma2_9b_it_v0.json"
MANIFEST = REPO / "configs/revision/ai_control/controlarena_apps_gemma2_9b_it_v0_source_manifest.json"
subprocess.run([sys.executable, str(REPO / "scripts/seal_controlarena_apps_gemma.py"), "--config", str(CONFIG), "--manifest", str(MANIFEST), "--verify"], check=True)

## 3. Persistent output directory

The three model views are cached separately, so an interrupted full run can resume without repeating completed inference.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTDIR = Path("/content/drive/MyDrive/observerbench/controlarena_apps_gemma2_9b_it_v0")
OUTDIR.mkdir(parents=True, exist_ok=True)
OUTDIR

## 4. The single compatibility smoke

Run this cell once. It audits all prompt lengths, checks the exact model architecture and digit-token contract, captures the three declared residual layers on two fit pairs, and encodes one row with each pinned official SAE. It does not inspect monitor quality.

In [ ]:
RUNNER = REPO / "scripts/run_controlarena_apps_gemma.py"
subprocess.run([
    sys.executable, str(RUNNER),
    "--config", str(CONFIG),
    "--manifest", str(MANIFEST),
    "--outdir", str(OUTDIR),
    "--stage", "compatibility-smoke",
    "--device", "cuda",
], check=True)

In [ ]:
import json
smoke = json.loads((OUTDIR / "compatibility_smoke.json").read_text())
{
    "status": smoke["status"],
    "model": smoke["model"],
    "locked_truncation": {name: item["at_or_above_limit_by_split"]["locked_test"] for name, item in smoke["length_audit"].items()},
    "saes": [{k: row[k] for k in ("layer", "d_in", "width", "encoded_shape", "finite")} for row in smoke["saes"]],
}

## 5. Review gate, then full run

Stop here for code and smoke review. Only after the smoke is accepted should `RUN_FULL` become `True`. The runner independently requires `--allow-full-run`, verifies the seal again, and verifies that the passing smoke used the same source bundle and APPS row identities.

In [ ]:
RUN_FULL = False
if not RUN_FULL:
    print("Full GPU run remains review-gated. No full run launched.")
else:
    subprocess.run([
        sys.executable, str(RUNNER),
        "--config", str(CONFIG),
        "--manifest", str(MANIFEST),
        "--outdir", str(OUTDIR),
        "--stage", "full",
        "--device", "cuda",
        "--resume",
        "--allow-full-run",
    ], check=True)

## 6. Result table

This cell is safe to run after completion. It displays the frozen two-percent audit comparison without changing any artifact.

In [ ]:
RESULT = OUTDIR / "results.json"
if RESULT.exists():
    result = json.loads(RESULT.read_text())
    primary = result["budget_results"]["0.02"]
    display({name: row["metrics"] for name, row in primary.items()})
else:
    print("No full-run result yet.")